<a href="https://colab.research.google.com/github/nayara-berti/safevoice-sst/blob/main/notebooks/safevoice_sst_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SafeVoice SST #

Assistente de voz para orientação preliminar em Segurança do Trabalho, com consulta a base normativa e documental.

1) Instalação

In [27]:
!pip install -q openai gTTS git+https://github.com/openai/whisper.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


2) Imports e idioma

In [28]:
language = "pt"

import whisper
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

print("Imports carregados com sucesso.")

Imports carregados com sucesso.


3) Função para gravar áudio no navegador

In [29]:
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  let chunks = []

  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()

  await sleep(time)

  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }

  recorder.stop()
})
"""

def record_audio(seconds=5, file_name="request_audio.wav"):
    display(Javascript(RECORD))
    js_result = output.eval_js(f"record({seconds * 1000})")
    audio_data = b64decode(js_result.split(",")[1])

    with open(file_name, "wb") as f:
        f.write(audio_data)

    return file_name

print("Função record_audio carregada com sucesso.")

Função record_audio carregada com sucesso.


4) Gravar e ouvir o áudio

In [39]:
print("Ouvindo...")
record_file = record_audio(seconds=10)

print("Áudio gravado com sucesso:")
display(Audio(record_file, autoplay=False))

Ouvindo...


<IPython.core.display.Javascript object>

Áudio gravado com sucesso:


5) Carregar o modelo Whisper

In [40]:
model = whisper.load_model("small")
print("Modelo Whisper carregado com sucesso.")

Modelo Whisper carregado com sucesso.


6) Transcrever o áudio

In [41]:
result = model.transcribe(
    record_file,
    fp16=False,
    language=language
)

transcription = result["text"].strip()

print("Transcrição:")
print(transcription)

Transcrição:
O que eu preciso fazer para acessar uma altura de 3 metros?


7) Base local de conhecimento

In [42]:
knowledge_base = {
    "altura": {
        "fonte": "NR-35",
        "resposta": """
A atividade mencionada envolve trabalho em altura, pois ocorre acima de 2 metros com risco de queda.
De forma orientativa, devem ser observados planejamento da atividade, análise de risco, medidas de prevenção,
condições de acesso, uso adequado de proteção contra quedas e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
    },
    "epi": {
        "fonte": "NR-06",
        "resposta": """
Em situações que envolvam riscos ocupacionais, devem ser observados os equipamentos de proteção individual adequados ao risco.
Também é importante verificar conservação, uso correto, orientação ao trabalhador e compatibilidade com a atividade.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
    },
    "incêndio": {
        "fonte": "NR-23",
        "resposta": """
Em temas relacionados à prevenção e combate a incêndio, devem ser observadas medidas de prevenção,
rotas de fuga, equipamentos de combate, sinalização e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
    },
    "espaço confinado": {
        "fonte": "NR-33",
        "resposta": """
Atividades em espaço confinado exigem identificação de riscos, controle de entrada, monitoramento da atmosfera,
procedimentos de emergência, capacitação e supervisão adequada.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
    },
    "máquinas": {
        "fonte": "NR-12",
        "resposta": """
Em atividades com máquinas e equipamentos, devem ser observados dispositivos de segurança,
proteções, procedimentos operacionais, capacitação e medidas de prevenção contra acidentes.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
    }
}

8) Função de resposta local

In [44]:
def generate_local_response(transcription):
    text = transcription.lower()

    for keyword, content in knowledge_base.items():
        if keyword in text:
            return content["resposta"].strip(), content["fonte"]

    default_response = """
Não foi possível identificar com precisão o tema normativo da pergunta.
Como orientação inicial, revise os riscos da atividade, consulte a Norma Regulamentadora aplicável
e valide a situação com análise técnica antes de executar qualquer atividade.
Esta resposta tem caráter preliminar e orientativo.
""".strip()

    return default_response, "Base local inicial"

9) Gerar resposta local

In [45]:
chatgpt_response, source_used = generate_local_response(transcription)

print("Fonte consultada:")
print(source_used)

print("\nResposta:")
print(chatgpt_response)

Fonte consultada:
NR-35

Resposta:
A atividade mencionada envolve trabalho em altura, pois ocorre acima de 2 metros com risco de queda.
De forma orientativa, devem ser observados planejamento da atividade, análise de risco, medidas de prevenção,
condições de acesso, uso adequado de proteção contra quedas e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.


10) Sintetizar a resposta com gTTS

In [46]:
response_audio = "response_audio.mp3"

gtts_object = gTTS(
    text=chatgpt_response,
    lang="pt",
    slow=False
)

gtts_object.save(response_audio)
print("Áudio da resposta salvo com sucesso.")

Áudio da resposta salvo com sucesso.


11) Ouvir a resposta

In [47]:
display(Audio(response_audio, autoplay=True))

**Versão Orientada a Objetos**

Depois que a versão acima funcionar (1 a 11), seguir com as póximas, executar apartir 12.

12) Classes do projeto

In [48]:
class SpeechToTextService:
    def __init__(self, model):
        self.model = model

    def transcribe_audio(self, audio_file, language="pt"):
        result = self.model.transcribe(
            audio_file,
            fp16=False,
            language=language
        )
        return result["text"].strip()


class KnowledgeBaseService:
    def __init__(self):
        self.knowledge_base = {
            "altura": {
                "fonte": "NR-35",
                "resposta": """
A atividade mencionada envolve trabalho em altura, pois ocorre acima de 2 metros com risco de queda.
De forma orientativa, devem ser observados planejamento da atividade, análise de risco, medidas de prevenção,
condições de acesso, uso adequado de proteção contra quedas e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
            },
            "epi": {
                "fonte": "NR-06",
                "resposta": """
Em situações que envolvam riscos ocupacionais, devem ser observados os equipamentos de proteção individual adequados ao risco.
Também é importante verificar conservação, uso correto, orientação ao trabalhador e compatibilidade com a atividade.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
            },
            "incêndio": {
                "fonte": "NR-23",
                "resposta": """
Em temas relacionados à prevenção e combate a incêndio, devem ser observadas medidas de prevenção,
rotas de fuga, equipamentos de combate, sinalização e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
            },
            "espaço confinado": {
                "fonte": "NR-33",
                "resposta": """
Atividades em espaço confinado exigem identificação de riscos, controle de entrada, monitoramento da atmosfera,
procedimentos de emergência, capacitação e supervisão adequada.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
            },
            "máquinas": {
                "fonte": "NR-12",
                "resposta": """
Em atividades com máquinas e equipamentos, devem ser observados dispositivos de segurança,
proteções, procedimentos operacionais, capacitação e medidas de prevenção contra acidentes.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.
"""
            }
        }

    def generate_local_response(self, transcription):
        text = transcription.lower()

        for keyword, content in self.knowledge_base.items():
            if keyword in text:
                return content["resposta"].strip(), content["fonte"]

        default_response = """
Não foi possível identificar com precisão o tema normativo da pergunta.
Como orientação inicial, revise os riscos da atividade, consulte a Norma Regulamentadora aplicável
e valide a situação com análise técnica antes de executar qualquer atividade.
Esta resposta tem caráter preliminar e orientativo.
""".strip()

        return default_response, "Base local inicial"


class TextToSpeechService:
    def generate_audio(self, text, output_file="response_audio.mp3", language="pt"):
        gtts_object = gTTS(
            text=text,
            lang=language,
            slow=False
        )
        gtts_object.save(output_file)
        return output_file


class SafeVoiceController:
    def __init__(self, stt_service, knowledge_service, tts_service):
        self.stt_service = stt_service
        self.knowledge_service = knowledge_service
        self.tts_service = tts_service

    def process_audio(self, audio_file, language="pt"):
        transcription = self.stt_service.transcribe_audio(audio_file, language)
        response_text, source_used = self.knowledge_service.generate_local_response(transcription)
        audio_output = self.tts_service.generate_audio(
            response_text,
            output_file="response_audio.mp3",
            language=language
        )

        return {
            "transcription": transcription,
            "response_text": response_text,
            "source_used": source_used,
            "audio_output": audio_output
        }

13) Instanciar os objetos

In [49]:
stt_service = SpeechToTextService(model)
knowledge_service = KnowledgeBaseService()
tts_service = TextToSpeechService()

controller = SafeVoiceController(
    stt_service=stt_service,
    knowledge_service=knowledge_service,
    tts_service=tts_service
)

print("Objetos criados com sucesso.")

Objetos criados com sucesso.


14) Rodar o fluxo completo com o controller

In [50]:
result = controller.process_audio(record_file, language="pt")

print("Transcrição:")
print(result["transcription"])

print("\nFonte consultada:")
print(result["source_used"])

print("\nResposta:")
print(result["response_text"])

Transcrição:
O que eu preciso fazer para acessar uma altura de 3 metros?

Fonte consultada:
NR-35

Resposta:
A atividade mencionada envolve trabalho em altura, pois ocorre acima de 2 metros com risco de queda.
De forma orientativa, devem ser observados planejamento da atividade, análise de risco, medidas de prevenção,
condições de acesso, uso adequado de proteção contra quedas e procedimentos de emergência.
Esta resposta tem caráter preliminar e não substitui avaliação técnica no local.


15) Ouvir a resposta da versão OO

In [51]:
display(Audio(result["audio_output"], autoplay=True))


Nesse projeto foi excluido a configuração de chave da OpenAI(API key)

Estrutura final do fluxo

gravar áudio → transcrever com Whisper → buscar resposta local → gerar áudio com gTTS → reproduzir